# Linear Regression from Scratch using PyTorch

In this notebook we build a simple linear regression model **from scratch** using PyTorch tensors and autograd, without relying on `torch.nn` or built-in optimizers. The goal is to understand what actually happens under the hood:

- Define a linear model `y = w * x + b`
- Compute predictions manually
- Define a Mean Squared Error (MSE) loss
- Use autograd to compute gradients
- Update parameters with manual gradient descent

After doing it from scratch, we briefly compare against the `torch.nn` approach.

## 1. Imports and setup

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 2. Generate synthetic data

We create data that follows a known linear relationship `y = 2x + 1` with a little noise, so we can check whether the model recovers the true parameters.

In [ ]:
# True parameters we want the model to learn
true_w, true_b = 2.0, 1.0

X = torch.linspace(0, 10, 100).reshape(-1, 1)
noise = torch.randn(X.shape) * 1.0
y = true_w * X + true_b + noise

plt.scatter(X.numpy(), y.numpy(), s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic data: y = 2x + 1 + noise")
plt.show()

## 3. Initialize parameters

We start the weight and bias at zero and tell PyTorch to track gradients for them with `requires_grad=True`.

In [ ]:
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

print(f"Initial w: {w.item():.4f}, b: {b.item():.4f}")

## 4. Define the model and loss

The model is just a linear function. The loss is Mean Squared Error (MSE).

In [ ]:
def model(x):
    return w * x + b

def mse_loss(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean()

## 5. Training loop (manual gradient descent)

On each step we:
1. Compute predictions and loss
2. Call `loss.backward()` to populate gradients
3. Update `w` and `b` inside `torch.no_grad()` so the update isn't tracked
4. Zero the gradients before the next step

In [ ]:
learning_rate = 0.001
epochs = 1000
losses = []

for epoch in range(epochs):
    # Forward pass
    y_pred = model(X)
    loss = mse_loss(y_pred, y)

    # Backward pass: compute gradients
    loss.backward()

    # Manual parameter update
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

        # Reset gradients for the next iteration
        w.grad.zero_()
        b.grad.zero_()

    losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1:4d} | loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

## 6. Results

Compare the learned parameters against the true ones, and plot the fitted line and loss curve.

In [ ]:
print(f"Learned w: {w.item():.4f} (true: {true_w})")
print(f"Learned b: {b.item():.4f} (true: {true_b})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(X.numpy(), y.numpy(), s=10, label="data")
with torch.no_grad():
    ax1.plot(X.numpy(), model(X).numpy(), color="red", label="fitted line")
ax1.set_title("Fitted line")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend()

ax2.plot(losses)
ax2.set_title("Training loss (MSE)")
ax2.set_xlabel("epoch")
ax2.set_ylabel("loss")

plt.tight_layout()
plt.show()

## 7. The same thing with `torch.nn`

For reference, here is the idiomatic PyTorch version using `nn.Linear` and a built-in optimizer. It does the same job with far less boilerplate.

In [ ]:
import torch.nn as nn

net = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = criterion(net(X), y)
    loss.backward()
    optimizer.step()

w_learned = net.weight.item()
b_learned = net.bias.item()
print(f"nn.Linear learned w: {w_learned:.4f}, b: {b_learned:.4f}")

## Summary

- A linear regression model is just `y = w * x + b`.
- PyTorch's autograd computes gradients for us once we call `loss.backward()`.
- Gradient descent updates each parameter in the direction that reduces the loss.
- Doing it from scratch makes clear what `torch.nn` and the optimizers automate for you.